# Vectyfi Radar - CAN Baseline (A/B Test)
### Goal:
Test the absolute predictive limit of the Contract Award Notice (CAN) dataset against our CFC baseline.
This notebook is explicitly designed to drop ALL "cheat codes" (e.g. Total Number of Offers, Exact Award Values) to test if the remaining pure structural CAN variables naturally skew predictions.

**Key constraints addressed:**
1. Focus entirely on `export_CAN_2023_2018.csv`
2. Removal of explicit Award metric 'cheat codes' to ensure a mathematically fair fight
3. Memory-efficient 500k sampling strategy\n

In [ ]:
# Cell 1: Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report

# Apply dark theme for visuals
plt.style.use('dark_background')
print('✅ Libraries loaded for CAN Baseline A/B test.')\n

In [ ]:
# Cell 2: Data Loading & Deduplication
file_path = '/Users/edu/Edu/testproject/tenderpilot_data/data/export_CAN_2023_2018.csv'

# Base features to keep
CAN_FEATURES = ['ID_NOTICE_CAN', 'CPV', 'ISO_COUNTRY_CODE', 'TYPE_OF_CONTRACT', 'B_FRA_AGREEMENT', 'CAE_TYPE']
# Target variables required strictly to generate 'y' (but MUST be dropped before X creation)
TARGET_COLS = ['WIN_NAME', 'AWARD_VALUE_EURO']
# Hard Cheats to eliminate immediately upon reading the file if we needed them, but we won't even load them
# 'NUMBER_OFFERS', 'INFO_ON_NON_AWARD', 'B_CONTRACTOR_SME', etc.

usecols = list(set(CAN_FEATURES + TARGET_COLS))

print(f"Loading selected CAN columns from {file_path} to prevent Out-Of-Memory...")
try:
    df_raw = pd.read_csv(file_path, usecols=usecols, low_memory=False)
except ValueError as e:
    print("Falling back to reading appropriate columns based on headers...")
    headers = pd.read_csv(file_path, nrows=0).columns.tolist()
    all_usecols = [c for c in usecols if c in headers]
    df_raw = pd.read_csv(file_path, usecols=all_usecols, low_memory=False)

print(f"Loaded {df_raw.shape[0]} CAN rows. Deduplicating on ID_NOTICE_CAN...")
df_dedup = df_raw.drop_duplicates(subset=['ID_NOTICE_CAN'], keep='first').copy()

print(f"Sampling 500k rows randomly for robust training...")
if len(df_dedup) > 500000:
    df_sampled = df_dedup.sample(n=500000, random_state=42).copy()
else:
    df_sampled = df_dedup.copy()

print(f"✅ Data prepared: {df_sampled.shape[0]} rows ready for target engineering.")\n

In [ ]:
# Cell 3: Target Engineering
print("Engineering target 'y' based on Final CAN outcomes...")

# If WIN_NAME is populated OR an AWARD_VALUE_EURO exists > 0, it means the CAN resulted in an actual contract mapping
if 'WIN_NAME' in df_sampled.columns and 'AWARD_VALUE_EURO' in df_sampled.columns:
    df_sampled['y'] = ((df_sampled['WIN_NAME'].notna()) | (df_sampled['AWARD_VALUE_EURO'] > 0)).astype(int)
else:
    raise ValueError("Target definition columns missing from the CAN dataset subset!")

print("\n=== Target Distribution ===")
print(df_sampled['y'].value_counts())
print(f"Award Rate in CAN dataset: {df_sampled['y'].mean()*100:.2f}%")
print("✅ Target engineered successfully.")\n

In [ ]:
# Cell 4: The Leakage Exorcism (Create X)
print("Performing the Leakage Exorcism: Dropping Target Links & Cheats...")

# The explicit variables that reveal the outcome
# These MUST go before X is established
LEAK_COLS = ['WIN_NAME', 'AWARD_VALUE_EURO', 'ID_NOTICE_CAN', 'y']
existing_cols_to_remove = [c for c in LEAK_COLS if c in df_sampled.columns]

X_raw = df_sampled.drop(columns=existing_cols_to_remove).copy()
y = df_sampled['y'].copy()

print(f"Dropped {len(existing_cols_to_remove)} outcome-link columns permanently.")
print("✅ Target leakage explicitly eliminated. This is a fair fight.")\n

In [ ]:
# Cell 5: Feature Engineering & Preprocessing
print("Engineering remaining CAN baseline features...")

X_processed = pd.DataFrame()

# Clean CPV if numeric/string
if 'CPV' in X_raw.columns:
    X_processed['cpv_group'] = X_raw['CPV'].astype(str).str[:2].replace('na', 'Unknown').fillna('Unknown')
    
if 'ISO_COUNTRY_CODE' in X_raw.columns:
    X_processed['country_code'] = X_raw['ISO_COUNTRY_CODE'].fillna('Unknown')
    
if 'B_FRA_AGREEMENT' in X_raw.columns:
    X_processed['b_fra_agreement'] = (X_raw['B_FRA_AGREEMENT'] == 'Y').astype(int)
    
if 'TYPE_OF_CONTRACT' in X_raw.columns:
    X_processed['type_of_contract'] = X_raw['TYPE_OF_CONTRACT'].fillna('Unknown')

if 'CAE_TYPE' in X_raw.columns:
    X_processed['cae_type'] = X_raw['CAE_TYPE'].astype(str).str[:1].replace('n', 'Unknown').fillna('Unknown')

# One-Hot Encoding for categoricals
print("Applying One-Hot Encoding to categorical features...")
cols_to_encode = [c for c in ['cpv_group', 'country_code', 'type_of_contract', 'cae_type'] if c in X_processed.columns]
X_encoded = pd.get_dummies(X_processed, 
                           columns=cols_to_encode, 
                           drop_first=True,
                           dtype=int)

print(f"Final feature space size: {X_encoded.shape[1]} columns")
print("✅ CAN features ready.")\n

In [ ]:
# Cell 6: Train/Test Split & XGBoost Training
print("Splitting data -> 80% Train, 20% Test")
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training XGBoost on {X_train.shape[0]} CAN samples...")
model = xgb.XGBClassifier(
    n_estimators=100, 
    max_depth=5, 
    learning_rate=0.1, 
    random_state=42,
    n_jobs=-1,
    use_label_encoder=False,
    eval_metric='auc'
)

model.fit(X_train, y_train)
print("✅ Model training finished successfully.")\n

In [ ]:
# Cell 7: Evaluation & Insights
# Generate predictions
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

# 1. ROC-AUC Score
auc_score = roc_auc_score(y_test, y_prob)
print("\n" + "="*40)
print(f"🏆 CAN Baseline ROC-AUC Score: {auc_score:.4f}")
print("="*40)

# Compare this to your CFC Notebook to declare the True Winner!

# 2. Confusion Matrix
print("\n--- Confusion Matrix ---")
cm = confusion_matrix(y_test, y_pred)
print(cm)

# 3. Feature Importance Plot
fig, ax = plt.subplots(figsize=(10, 6))
importances = pd.Series(model.feature_importances_, index=X_train.columns)
top_10 = importances.sort_values(ascending=True).tail(10)

# Bright magenta highlighting for the CAN model
top_10.plot(kind='barh', ax=ax, color='#ff007f', edgecolor='#202020')

ax.set_title('Top 10 CAN Baseline Features (Vectyfi Radar A/B Test)', fontsize=15, fontweight='bold', color='white')
ax.set_xlabel('Relative Importance (Gain)', color='white')
ax.set_ylabel('Features', color='white')
plt.tight_layout()

print("✅ Evaluation complete. Displaying Visuals:")
plt.show()\n